# Machine Learning Models Are No Longer a Black Box

Complete companion for [the article](https://todatabeyond.com/blog/machine-learning-models-are-no-longer-a-black-box): permutation importance, partial dependence, SHAP and LIME on the original automobile price regression problem.

Run all cells from a fresh Python 3.11+ kernel. CPU only; no keys. The official UCI Automobile download is checksum-verified. All 205 source rows are loaded; four rows without a price are removed, as in the author's original preprocessing notebook. All 201 remaining rows and twelve encoded columns are used. This intentionally retains the source's same-data model inspection; its R² and explanations are not held-out performance estimates.

The missing setup/preprocessing and promised feature-mean/bar-chart steps are supplied. Original gist snapshots and revisions remain below. Runtime-dependent values and original screenshots are not interchangeable.


In [ ]:
import subprocess
import sys
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check",
                "numpy==2.3.3", "scipy==1.16.2", "pandas==2.3.3",
                "scikit-learn==1.7.2", "matplotlib==3.10.9", "eli5==0.16.0",
                "shap==0.49.1", "lime==0.2.0.1", "certifi==2026.10.23",
                "ipywidgets==8.1.7"], check=True)
import hashlib
import io
import os
import ssl
import urllib.request
import zipfile
from pathlib import Path
import importlib.metadata as metadata
import certifi
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
np.set_printoptions(precision=6, suppress=True)
pd.set_option("display.max_rows", 20)
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 15)
print("Python:", sys.version.split()[0])
print("Libraries:", ", ".join(f"{p} {metadata.version(p)}" for p in ["numpy", "pandas", "scikit-learn", "eli5", "shap", "lime"]))

## Original automobile data preparation

Data: [UCI Automobile](https://archive.ics.uci.edu/dataset/10/automobile), Jeffrey Schlimmer, 1987, [DOI](https://doi.org/10.24432/C5B01C), CC BY 4.0. Same 205-row imports-85 data underlying the author's automobile notebook. The original notebook is pinned to commit `63fac8eec9a1106cb9de3c75d7bffd3d9a97db3f`.

Reproduce the relevant original preprocessing in order: mean-impute bore and horsepower over all source rows, remove missing prices, normalize length/width by their post-drop maxima, truncate horsepower to integer as the author did. Other original notebook EDA/unused columns are not required by these ten selected features.


In [ ]:
DATA_URL = "https://archive.ics.uci.edu/static/public/10/automobile.zip"
ZIP_SHA256 = "0360d2379d2d505540f0c43bfe0ada96244ad2662faf9dada1629824948294ec"
DATA_SHA256 = "125db8bd36a84801f7223faf54071d15f85f26c13a9a7aee63375ac46098fb7f"
cached_zip = os.environ.get("TDB_AUTOMOBILE_ZIP")
if cached_zip:
    archive_bytes = Path(cached_zip).read_bytes()
else:
    tls_context = ssl.create_default_context(cafile=certifi.where())
    with urllib.request.urlopen(DATA_URL, context=tls_context, timeout=60) as response:
        archive_bytes = response.read()
assert hashlib.sha256(archive_bytes).hexdigest() == ZIP_SHA256, "Unexpected UCI archive contents"
with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
    csv_bytes = archive.read("automobile/imports-85.data")
assert hashlib.sha256(csv_bytes).hexdigest() == DATA_SHA256, "Unexpected automobile data"
headers = ["symboling", "normalized-losses", "make", "fuel-type", "aspiration", "num-of-doors",
           "body-style", "drive-wheels", "engine-location", "wheel-base", "length", "width",
           "height", "curb-weight", "engine-type", "num-of-cylinders", "engine-size", "fuel-system",
           "bore", "stroke", "compression-ratio", "horsepower", "peak-rpm", "city-mpg", "highway-mpg", "price"]
raw_df = pd.read_csv(io.BytesIO(csv_bytes), names=headers, na_values="?")
df = raw_df.copy()
for column in ["bore", "horsepower"]:
    df[column] = df[column].fillna(df[column].mean())
df = df.dropna(subset=["price"]).reset_index(drop=True)
for column in ["length", "width"]:
    df[column] = df[column] / df[column].max()
df["horsepower"] = df["horsepower"].astype(int)
print("Data SHA-256:", hashlib.sha256(csv_bytes).hexdigest())
print("Original / priced rows:", len(raw_df), "/", len(df))
print("Mean engine size:", f"{df['engine-size'].mean():.8f}")

## 1. Permutation importance

In [ ]:
import eli5
from eli5.sklearn import PermutationImportance
print("ELI5 permutation importance ready.")

In [ ]:
important_features = ["length", "width", "curb-weight", "engine-size", "horsepower",
                      "city-mpg", "highway-mpg", "wheel-base", "bore", "drive-wheels"]
X = pd.get_dummies(df[important_features], columns=["drive-wheels"], dtype=float)
y = df["price"]
X_array = X.to_numpy(dtype=float)
y_array = y.to_numpy(dtype=float)
# Fit/predict consistently on numerical arrays; retain X's feature names for explanations.
lm = LinearRegression().fit(X_array, y_array)
def predict_prices(rows):
    return lm.predict(np.asarray(rows, dtype=float))
print("Encoded features:", X.shape)
print("In-sample R²:", f"{lm.score(X_array, y_array):.8f}")
print("First two predictions:", predict_prices(X.iloc[:2]))

In [ ]:
perm = PermutationImportance(lm, random_state=1, n_iter=5, scoring="r2").fit(X_array, y_array)
importance_table = pd.DataFrame({"feature": X.columns,
                                 "mean_R2_decrease": perm.feature_importances_,
                                 "std_R2_decrease": perm.feature_importances_std_})
importance_table = importance_table.sort_values("mean_R2_decrease", ascending=False, kind="stable")
print(importance_table.to_string(index=False, float_format=lambda value: f"{value:.6f}"))

## 2. Partial dependence

In [ ]:
from sklearn.inspection import PartialDependenceDisplay
engine_index = X.columns.get_loc("engine-size")
fig, ax = plt.subplots(figsize=(8, 5))
pdp_display = PartialDependenceDisplay.from_estimator(
    lm, X_array, [engine_index], feature_names=X.columns.tolist(),
    method="brute", grid_resolution=100, ax=ax)
ax.set_title("PDP of the engine-size feature")
fig.tight_layout()
print("Engine-size grid points:", len(pdp_display.pd_results[0]["grid_values"][0]))
print("Partial-dependence endpoints:", pdp_display.pd_results[0]["average"][0, [0, -1]])
plt.show()

## 3. SHAP

Use model-aware `shap.Explainer(lm, X)` for this linear model instead of a much slower callable-based permutation explainer. The default independent masker keeps the same deterministic maximum-100-row background policy as the original call. Check its actual background and exact linear SHAP formula in the regression cell. These are interventional feature attributions, not causal effects; correlated and mutually exclusive one-hot features need thoughtful interpretation.


In [ ]:
import shap
shap_explainer = shap.Explainer(lm, X)
shap_values = shap_explainer(X)
sample_ind = 1
shap.plots.waterfall(shap_values[sample_ind], max_display=14, show=False)
plt.gcf().set_size_inches(10, 7)
plt.tight_layout()
print("Explained rows/features:", shap_values.values.shape)
print("SHAP background rows:", len(shap_explainer.masker.data))
print("Second-row base / prediction:", f"{shap_values.base_values[sample_ind]:.6f}", "/", f"{predict_prices(X.iloc[[sample_ind]])[0]:.6f}")
print("SHAP contribution sum:", f"{shap_values.values[sample_ind].sum():.6f}")
plt.show()

In [ ]:
print(X.mean().to_string(float_format=lambda value: f"{value:.6f}"))

In [ ]:
np.random.seed(42)
shap.plots.beeswarm(shap_values, max_display=14, show=False)
plt.gcf().set_size_inches(10, 7)
plt.tight_layout()
mean_abs_shap = pd.Series(np.abs(shap_values.values).mean(axis=0), index=X.columns).sort_values(ascending=False)
print("Mean absolute SHAP value:")
print(mean_abs_shap.to_string(float_format=lambda value: f"{value:.6f}"))
plt.show()

## 4. LIME

Installation is already pinned in setup. Retain the source's discretized tabular regression explainer, seed 42 and first row (`idx=0`). Its local neighborhood samples one-hot columns independently, so those perturbations are not guaranteed valid automobiles. The local surrogate is approximate and is not the original model or a global explanation. Print the same contribution table as static text rather than embedding JavaScript/HTML; retain the original source screenshot in the article. The promised `as_pyplot_figure()` line, absent from the original gists, is supplied.


In [ ]:
assert metadata.version("lime") == "0.2.0.1"
print("LIME", metadata.version("lime"), "is installed by the setup cell.")

In [ ]:
from lime.lime_tabular import LimeTabularExplainer
lime_explainer = LimeTabularExplainer(
    X_array, mode="regression", feature_names=X.columns.tolist(), random_state=42)
print("LIME mode:", lime_explainer.mode)
print("Neighborhood samples:", 5000)

In [ ]:
idx = 0
explanation = lime_explainer.explain_instance(
    X.iloc[idx].to_numpy(dtype=float), predict_prices,
    num_features=len(X.columns), num_samples=5000)
lime_table = pd.DataFrame(explanation.as_list(), columns=["condition", "contribution"])
print(lime_table.to_string(index=False, float_format=lambda value: f"{value:.6f}"))
print("Original model prediction:", f"{explanation.predicted_value:.6f}")
print("Local surrogate prediction:", f"{explanation.local_pred[0]:.6f}")
print("Local surrogate weighted R²:", f"{explanation.score:.6f}")

In [ ]:
fig = explanation.as_pyplot_figure()
fig.set_size_inches(10, 7)
fig.axes[0].set_title("LIME contributions for the first automobile")
fig.tight_layout()
print("Plotted contribution bars:", len(fig.axes[0].patches))
plt.show()

## Independent regression checks

In [ ]:
assert raw_df.shape == (205, 26) and X.shape == (201, 12)
assert raw_df["price"].isna().sum() == 4
assert np.isfinite(X_array).all() and np.isfinite(y_array).all()
assert X.columns.tolist()[-3:] == ["drive-wheels_4wd", "drive-wheels_fwd", "drive-wheels_rwd"]
np.testing.assert_array_equal(X_array[:, -3:].sum(axis=1), np.ones(201))
np.testing.assert_allclose(df["engine-size"].mean(), 126.87562189054727)
# Solve the rank-deficient one-hot design independently; coefficients may differ but predictions agree.
design = np.column_stack([np.ones(len(X_array)), X_array])
ols_coefficients = np.linalg.lstsq(design, y_array, rcond=None)[0]
np.testing.assert_allclose(design @ ols_coefficients, predict_prices(X_array), rtol=1e-8, atol=1e-6)
# ELI5 shuffles each restored column using a single seeded RNG, five full rounds.
rng = np.random.RandomState(1)
baseline = 1 - np.sum((y_array - predict_prices(X_array)) ** 2) / np.sum((y_array - y_array.mean()) ** 2)
manual_importances = []
for repeat in range(5):
    decreases = []
    for column in range(X_array.shape[1]):
        shuffled = X_array.copy()
        rng.shuffle(shuffled[:, column])
        r2 = 1 - np.sum((y_array - predict_prices(shuffled)) ** 2) / np.sum((y_array - y_array.mean()) ** 2)
        decreases.append(baseline - r2)
    manual_importances.append(decreases)
np.testing.assert_allclose(perm.results_, manual_importances, rtol=1e-10, atol=1e-12)
np.testing.assert_allclose(perm.feature_importances_, np.mean(manual_importances, axis=0), atol=1e-12)
np.testing.assert_allclose(perm.feature_importances_std_, np.std(manual_importances, axis=0), atol=1e-12)
# Every PDP grid prediction is the average prediction with engine size replaced.
pdp_grid = pdp_display.pd_results[0]["grid_values"][0]
manual_pdp = []
for value in pdp_grid:
    changed = X_array.copy(); changed[:, engine_index] = value
    manual_pdp.append(predict_prices(changed).mean())
np.testing.assert_allclose(pdp_display.pd_results[0]["average"][0], manual_pdp, rtol=1e-12, atol=1e-8)
np.testing.assert_allclose(np.diff(manual_pdp), np.diff(pdp_grid) * lm.coef_[engine_index], atol=1e-8)
# Exact interventional linear SHAP formula and additivity for every row/feature.
background_mean = shap_explainer.masker.data.mean(axis=0)
manual_shap = (X_array - background_mean) * lm.coef_
manual_base = lm.intercept_ + background_mean @ lm.coef_
np.testing.assert_allclose(shap_values.values, manual_shap, rtol=1e-12, atol=1e-8)
np.testing.assert_allclose(shap_values.base_values, manual_base, atol=1e-8)
np.testing.assert_allclose(shap_values.base_values + shap_values.values.sum(axis=1), predict_prices(X_array), atol=1e-8)
# Capture LIME's actual perturbed local design and rebuild its weighted ridge surrogate independently.
captured_local_fit = {}
repeat_explainer = LimeTabularExplainer(X_array, mode="regression", feature_names=X.columns.tolist(), random_state=42)
# Attach capture to the identically seeded explainer.
repeat_original_fit = repeat_explainer.base.explain_instance_with_data
def capture_repeat(neighborhood_data, neighborhood_labels, distances, label, num_features, **kwargs):
    captured_local_fit.update(data=neighborhood_data.copy(), labels=neighborhood_labels[:, label].copy(), distances=distances.copy())
    return repeat_original_fit(neighborhood_data, neighborhood_labels, distances, label, num_features, **kwargs)
repeat_explainer.base.explain_instance_with_data = capture_repeat
repeat_explanation = repeat_explainer.explain_instance(X_array[idx], predict_prices, num_features=12, num_samples=5000)
np.testing.assert_allclose(repeat_explanation.local_pred, explanation.local_pred, atol=1e-10)
local_x = captured_local_fit["data"]; local_y = captured_local_fit["labels"]
weights = repeat_explainer.base.kernel_fn(captured_local_fit["distances"])
assert local_x.shape == (5000, 12) and np.isfinite(local_x).all()
x_mean = np.average(local_x, axis=0, weights=weights); y_mean = np.average(local_y, weights=weights)
centered_x = local_x - x_mean; centered_y = local_y - y_mean
ridge_coef = np.linalg.solve(centered_x.T @ (weights[:, None] * centered_x) + np.eye(12), centered_x.T @ (weights * centered_y))
ridge_intercept = y_mean - x_mean @ ridge_coef
actual_coef = dict(explanation.local_exp[1])
np.testing.assert_allclose([actual_coef[i] for i in range(12)], ridge_coef, atol=1e-7)
np.testing.assert_allclose(explanation.intercept[1], ridge_intercept, atol=1e-7)
np.testing.assert_allclose(explanation.local_pred[0], ridge_intercept + local_x[0] @ ridge_coef, atol=1e-7)
weighted_residual = np.sum(weights * (local_y - (ridge_intercept + local_x @ ridge_coef)) ** 2)
weighted_total = np.sum(weights * (local_y - y_mean) ** 2)
np.testing.assert_allclose(explanation.score, 1 - weighted_residual / weighted_total, atol=1e-10)
repeat_explainer.base.explain_instance_with_data = repeat_original_fit
print("All 205 source rows / 201 priced rows and twelve encoded features verified.")
print("Independent OLS predictions, all sixty permutation shuffles and every PDP grid value agree.")
print("All 2412 SHAP values and 201 additive prediction reconstructions agree.")
print("LIME seed replay, all twelve weighted-ridge coefficients, intercept, local prediction and weighted R² agree.")

## Maintenance and interpretation

- Repair the incomplete `y = df['price` assignment; provide the missing LinearRegression/pandas imports and data setup.
- Preserve source features, full-data fit, ELI5 seed and five permutation rounds. R² is the estimator's regression score, not classification accuracy. Correlated variables can redistribute apparent importance.
- Create the PDP figure before plotting, remove the undefined standalone `figure()` call, use a named engine-size index and save the same estimator's results for numerical checks.
- Use the model-aware linear SHAP implementation with the same default 100-row independent background policy. Preserve row 1 for its waterfall, and all rows for the beeswarm. Supply the promised feature means. Match values to the actual masker, not to an assumed full-data mean.
- Preserve first-row LIME (`idx=0`), not the source prose's claimed second row. Print static contributions instead of JavaScript HTML; add the missing bar-chart call. Quantify approximation using the actual surrogate's local prediction and weighted fit, not an equality claim.
- No causal, legal-compliance, clinical-decision or credit-decision guarantees are established by these diagrams. The source's lending/health examples are illustrative, not validated deployment claims.
- No held-out split is introduced: this is source-faithful inspection, not evidence of generalization. Feature preprocessing uses the whole data in the original order. Do not use these in-sample metrics as an unbiased test score.

Primary docs: [ELI5](https://eli5.readthedocs.io/en/latest/blackbox/permutation_importance.html), [PDP](https://scikit-learn.org/1.7/modules/partial_dependence.html), [SHAP](https://shap.readthedocs.io/en/latest/generated/shap.LinearExplainer.html), [LIME](https://lime-ml.readthedocs.io/en/latest/lime.html).


## Immutable original gist appendix (not executable)

### 42b1e36541c4edbd459ab8805ce06121 / b803a7c873e1266bcd26af85805fc8a265825ccc

Original filename: importing permutation importance library

```python
import eli5
from eli5.sklearn import PermutationImportance
```

### f1fe2202439cd07f066d51aec83d5e36 / c93d70ab6af7bda3b93d10cd87624ef373138f2e

Original filename: loading and training the data

```python
# defining the important features 
important_features = ['length',
                      'width',
                      'curb-weight',
                      'engine-size',
                      'horsepower',
                      'city-mpg',
                      'highway-mpg',
                      'wheel-base',
                      'bore',
                      'drive-wheels']


X = df[important_features]
X = pd.get_dummies(X, 'drive-wheels')  # one hot encoding for the categorical features 
y = df['price

# Fitting linear regression model
lm = LinearRegression()
lm.fit(X, y)
```

### def6598b909a1af35b04fac37d92ef77 / 7d536a9c46a2d678be3f6fceae66a4c706b31b37

Original filename: fitting the permutation importance function

```python
perm = PermutationImportance(lm, random_state=1).fit(X, y)
eli5.show_weights(perm, feature_names = X.columns.tolist())
```

### d0a870e84d9bcf548c0b45c5a14340f9 / bc4b4785ee12f7608d2d6120dc8edb23230913ff

Original filename: PDP for engine-size

```python
from matplotlib import pyplot as plt
from sklearn.inspection import PartialDependenceDisplay

features = [3] # The engine size feature 
PartialDependenceDisplay.from_estimator(lm, X, features) 
plt.title('PDP of the engine-size feature')
figure(figsize=(10, 6), dpi=80)
```

### 65a66cb9c657dfc2d0d31dc88c3ddaae / ec473c933ae77d184329c27ce852ecd8b49ae212

Original filename: Plot Shap value for second row

```python
import shap  # package used to calculate Shap values

# compute the SHAP values for the linear regression model
explainer = shap.Explainer(lm.predict, X)
shap_values = explainer(X)


# the waterfall_plot shows how we get from shap_values.base_values to model.predict(X)[sample_ind]
sample_ind = 1 # the row index 
shap.plots.waterfall(shap_values[sample_ind], max_display=14)
```

### 39558a983ebb745f7fd2b062aa951db0 / 3d35b2bd37c4cf0294597f77b7e285bac7e8793d

Original filename: SHAP summary plot

```python
shap.plots.beeswarm(shap_values, max_display=14)
```

### c7addce4f17a1a028449930dd6a8fed7 / ec28e70acd745e4bbad1c86ead382207a118b438

Original filename: Installing the lime package

```python
pip install lime
```

### ee06142058b9c15894d7c05994a37894 / 36314449fc1ea2be02cea223dea28e5334acece1

Original filename: Create LIME explainer

```python
from lime.lime_tabular import LimeTabularExplainer
explainer = LimeTabularExplainer(
                   X.values,
                   mode="regression",
                   feature_names=X.columns,
                   random_state=42)
```

### 785cc6c032aa34b2fd3945241c69965a / 0c19a17f5c50b8e154694b07f1512115e7518a6d

Original filename: Create the LIME plots

```python
idx = 0
explanation = explainer.explain_instance(X.iloc[idx,:], lm.predict, num_features=len(X.columns))
explanation.show_in_notebook()
```

